In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import vgg19, VGG19_Weights
import math


from s2flow.data.pca import PCAConvLayer
from s2flow.utils import get_device

In [32]:
class MultispectralPerceptualLoss(nn.Module):
    """
    Perceptual loss wrapper that projects multispectral input (e.g., 4-band) 
    to 3-band RGB using a PCA layer before computing VGG features.
    """
    def __init__(self, config):
        super(MultispectralPerceptualLoss, self).__init__()
        self.device = get_device()
        
        # Initialize PCA Layer for dimensionality reduction (4 -> 3)
        self.pca_layer = PCAConvLayer(config).to(self.device)
        self.k = config.get('metrics', {}).get('pca_lpips_k', 3.0)
        self.clamp = config.get('metrics', {}).get('pca_lpips_clamp', True)

        # Load VGG19 features
        vgg = vgg19(weights=VGG19_Weights.DEFAULT)
        self.features = nn.ModuleList(list(vgg.features)).eval()
        
        # Layer indices for "conv1_2", "conv2_2", "conv3_4", "conv4_4", "conv5_4"
        # Adjusted slightly to match standard implementation of "before activation"
        self.layer_indices = {
            'conv1': 2, 'conv2': 7, 'conv3': 16, 'conv4': 25, 'conv5': 34
        }
        self.weights = {'conv1': 0.1, 'conv2': 0.1, 'conv3': 1.0, 'conv4': 1.0, 'conv5': 1.0}

        # Freeze VGG parameters
        for param in self.parameters():
            param.requires_grad = False
            
    def forward(self, pred, target):
        """
        Args:
            pred: (B, C_in, H, W)
            target: (B, C_in, H, W)
        """
        # 1. Project to 3-channel using PCA (differentiable)
        # Note: PCAConvLayer expects standard forward, ensuring gradients flow back to Generator
        x_feat = self.pca_layer(pred, k=self.k, clamp=self.clamp)
        y_feat = self.pca_layer(target, k=self.k, clamp=self.clamp)
        
        loss = 0
        current_layer = 0
        for name, index in sorted(self.layer_indices.items(), key=lambda item: item[1]):
            for i in range(current_layer, index):
                x_feat = self.features[i](x_feat)
                y_feat = self.features[i](y_feat)
            
            x_feat = self.features[index](x_feat)
            y_feat = self.features[index](y_feat)
            
            loss += self.weights[name] * F.l1_loss(x_feat, y_feat)
            current_layer = index + 1
            
        return loss

In [33]:
x = torch.randn(2, 4, 64, 64).to('cuda')
percep_loss = MultispectralPerceptualLoss({}).to('cuda')
y = torch.randn(2, 4, 64, 64).to('cuda')
loss = percep_loss(x, y)

In [ ]:
def make_layer(block, n_layers):
    layers = []
    for _ in range(n_layers):
        layers.append(block())
    return nn.Sequential(*layers)

    
class SRGANDiscriminator(nn.Module):
    """
    Standard VGG-Style Discriminator (For SRGAN and ESRGAN).
    """
    def __init__(self, config):
        super(SRGANDiscriminator, self).__init__()
        disc_conf = config.get('discriminator_model', {})
        in_channels = disc_conf.get('in_channels', 4)
        num_feat = disc_conf.get('num_feat', 64)

        def discriminator_block(in_filters, out_filters, first_block=False):
            layers = []
            layers.append(nn.Conv2d(in_filters, out_filters, 3, 1, 1, bias=False))
            if not first_block:
                layers.append(nn.BatchNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            layers.append(nn.Conv2d(out_filters, out_filters, 3, 2, 1, bias=False))
            layers.append(nn.BatchNorm2d(out_filters))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        layers = []
        layers.extend(discriminator_block(in_channels, num_feat, first_block=True)) # 64
        layers.extend(discriminator_block(num_feat, num_feat * 2))     # 128
        layers.extend(discriminator_block(num_feat * 2, num_feat * 4)) # 256
        layers.extend(discriminator_block(num_feat * 4, num_feat * 8)) # 512

        self.features = nn.Sequential(*layers)
        
        # Adaptive pooling allows variable input sizes
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(num_feat * 8, 1024),
            nn.LeakyReLU(0.2, True),
            nn.Linear(1024, 1)
        )

    def forward(self, x):
        out = self.features(x)
        return self.classifier(out)


In [52]:
generator_model = RRDBNet({}).to('cuda')
discriminator_model = SRGANDiscriminator({}).to('cuda')

# discriminator_model
generator_model

RRDBNet(
  (conv_first): Conv2d(4, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (body): Sequential(
    (0): RRDB(
      (rdb1): ResidualDenseBlock_RRDB(
        (conv1): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(96, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv3): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv4): Conv2d(160, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv5): Conv2d(192, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (lrelu): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (rdb2): ResidualDenseBlock_RRDB(
        (conv1): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(96, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv3): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv4): Conv2d(160, 32, kernel_size=(3, 3), strid

In [53]:
x = torch.randn(2, 4, 64, 64).to('cuda')
y_hat = generator_model(x)
y_hat.shape

torch.Size([2, 4, 256, 256])

In [57]:
from s2flow.models import UNetTensorWrapper

In [59]:
model = UNetTensorWrapper({})

In [60]:
model

UNetTensorWrapper(
  (model): UNet2DModel(
    (conv_in): Conv2d(8, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (time_proj): Timesteps()
    (time_embedding): TimestepEmbedding(
      (linear_1): Linear(in_features=64, out_features=256, bias=True)
      (act): SiLU()
      (linear_2): Linear(in_features=256, out_features=256, bias=True)
    )
    (down_blocks): ModuleList(
      (0): DownBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 64, eps=1e-05, affine=True)
            (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (time_emb_proj): Linear(in_features=256, out_features=64, bias=True)
            (norm2): GroupNorm(32, 64, eps=1e-05, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): Mod